# CV - Image Classification - Segmentation - Autoencoders

Note that this lab has three levels: basic, regular and advanced.


Doing the **basic** part earns you a grade of 5.5-6.0.

Doing the **regular** part earns you a max grade of 8.0.

Doing the **advanced** part earns you a max grade of 10.0.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.

## Basic Level

Exploring the basic concepts discussed in the lecture.


In [1]:
import numpy as np
import re
import keras
from sklearn.model_selection import train_test_split
from keras.datasets import cifar10, mnist, cifar100
from keras.models import Sequential
from keras.layers import Input, Dense, Activation,BatchNormalization, Flatten, Conv2D, MaxPooling2D
from sklearn import preprocessing
import matplotlib.pyplot as plt
from keras.applications import EfficientNetV2S, ResNet50
from keras.utils import to_categorical 
import pandas as pd
import numpy as np
from PIL import Image
import os
import os.path
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
from matplotlib import pyplot as plt
import keras
from keras.metrics import MeanIoU
from keras.utils import to_categorical 
import random
from sklearn.metrics import confusion_matrix
from scipy.special import softmax
from keras.layers import LeakyReLU
from sklearn.utils import shuffle
from keras.layers import  Input, Dense, BatchNormalization, Conv2D, Dropout, Conv2DTranspose, Concatenate
from keras import layers
from keras.layers import Input, Dense, Activation,BatchNormalization, Flatten, Conv2D, MaxPooling2D
from keras.models import Model
import keras.backend as K
from skimage import io
from skimage.util import random_noise
from skimage.transform import rotate, AffineTransform,warp

from keras import ops
from keras.applications.resnet50 import preprocess_input


I0000 00:00:1791204812.269520  929615 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791204812.276313  929615 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791204812.660957  929615 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1791204815.477719  929615 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791204815.479917  929615 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


E0000 00:00:1791204819.188129  929615 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Classification and Transfer Learning

### Exercise 1

In this exercise, we will work with the CIFAR100 dataset. 

The dataset can be import from keras datasets directly. No preprocessing is needed, you can directly use it to learn a model.

The CIFAR100 dataset has a 100 classes, each with 600 examples.

a. Let's load the dataset from keras

In [3]:
(train_X, train_y), (test_X, test_y)=cifar100.load_data()

In [4]:
train_y=to_categorical(train_y)

In [5]:
test_y=to_categorical(test_y)

b. Let's create a CNN model to learn to classify the data.

Create your model using the functional API.

In [6]:
keras.utils.set_random_seed(42)

def build_cnn(n_classes, name):
    inputs = Input(shape=(32, 32, 3))
    x = layers.Rescaling(1 / 255)(inputs)
    x = Conv2D(32, 3, padding='same', activation='relu')(x)
    x = MaxPooling2D()(x)
    x = Conv2D(64, 3, padding='same', activation='relu')(x)
    x = MaxPooling2D()(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(n_classes, activation='softmax')(x)
    return Model(inputs, outputs, name=name)

cnn_basic = build_cnn(100, 'cnn_basic')
cnn_basic.summary()

Model: "cnn_basic"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        12,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 556,708 (2.12 MB)

 Trainable params: 556,708 (2.12 MB)

 Non-trainable params: 0 (0.00 B)

c. compile and fit the model to the data.

This process might take too long, set the number of epochs to some low value. The goal of this part is to show that this classification problem (100 classes) is complex.

In [7]:
cnn_basic.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [8]:
history_100 = cnn_basic.fit(train_X, train_y, epochs=5, batch_size=64, validation_data=(test_X, test_y))

Epoch 1/5

  1/782 ━━━━━━━━━━━━━━━━━━━━ 18:58 1s/step - accuracy: 0.0156 - loss: 4.6047

  3/782 ━━━━━━━━━━━━━━━━━━━━ 34s 44ms/step - accuracy: 0.0260 - loss: 4.6088

  5/782 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.0219 - loss: 4.6162

  8/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.0176 - loss: 4.6126

 10/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.0188 - loss: 4.6107

...

774/782 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4308 - loss: 2.1960

776/782 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4310 - loss: 2.1952

778/782 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4312 - loss: 2.1952

780/782 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4313 - loss: 2.1948

782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 30ms/step - accuracy: 0.4312 - loss: 2.1947 - val_accuracy: 0.3781 - val_loss: 2.4777


### Exercise 2

The model trained on CIFAR100 is probably not giving the best results you can achieve. 

Let's train the same model on CIFAR10, a version of CIFAR with only 10 classes instead of a 100.

In [9]:
(train_X, train_y), (test_X, test_y)=cifar10.load_data()

In [10]:
train_y=to_categorical(train_y)
test_y=to_categorical(test_y)

a. Recreate your basic model here, compile it and fit it to the data.

In [11]:
cnn_10 = build_cnn(10, 'cnn_cifar10')
cnn_10.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_10 = cnn_10.fit(train_X, train_y, epochs=5, batch_size=64, validation_data=(test_X, test_y))

Epoch 1/5

  1/782 ━━━━━━━━━━━━━━━━━━━━ 12:36 969ms/step - accuracy: 0.1094 - loss: 2.2872

  4/782 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.1367 - loss: 2.2836   

  6/782 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - accuracy: 0.1172 - loss: 2.2985

  8/782 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - accuracy: 0.1367 - loss: 2.2768

 10/782 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.1312 - loss: 2.2718

...

772/782 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7394 - loss: 0.7490

775/782 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7394 - loss: 0.7490

778/782 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7396 - loss: 0.7485

780/782 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7397 - loss: 0.7485

782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.7397 - loss: 0.7482 - val_accuracy: 0.6840 - val_loss: 0.9359


### Exercise 3

Our basic CNN model does better on the CIFAR10 dataset than it did on the CIFAR100. 

The CIFAR10 is less comples, distinguishing 10 classes is a lot less complex than distinguishing a 100.

We can use transfer learning to solve this complex problem. We'll reuse the ResNet50 pretrained model from Keras, and adapt it to work on the CIFAR100.

a. Load Resnet50 from keras. Use the **imagenet** weights.

You can refer to keras to learn how to use this pretrained model, [here](https://keras.io/api/applications/resnet/).

In [12]:
(train_X, train_y), (test_X, test_y) = cifar100.load_data()
train_y = to_categorical(train_y)
test_y = to_categorical(test_y)

pretrained = ResNet50(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

In [13]:
#Showing the layers names and inputs
for i, layer in enumerate(pretrained.layers):
    print(i, layer.name, layer.output.shape)

0 input_layer_2 (None, 64, 64, 3)
1 conv1_pad (None, 70, 70, 3)
2 conv1_conv (None, 32, 32, 64)
3 conv1_bn (None, 32, 32, 64)
4 conv1_relu (None, 32, 32, 64)
5 pool1_pad (None, 34, 34, 64)
6 pool1_pool (None, 16, 16, 64)
7 conv2_block1_1_conv (None, 16, 16, 64)
8 conv2_block1_1_bn (None, 16, 16, 64)
9 conv2_block1_1_relu (None, 16, 16, 64)
10 conv2_block1_2_conv (None, 16, 16, 64)
11 conv2_block1_2_bn (None, 16, 16, 64)
12 conv2_block1_2_relu (None, 16, 16, 64)
13 conv2_block1_0_conv (None, 16, 16, 256)
14 conv2_block1_3_conv (None, 16, 16, 256)
15 conv2_block1_0_bn (None, 16, 16, 256)
16 conv2_block1_3_bn (None, 16, 16, 256)
17 conv2_block1_add (None, 16, 16, 256)
18 conv2_block1_out (None, 16, 16, 256)
19 conv2_block2_1_conv (None, 16, 16, 64)
20 conv2_block2_1_bn (None, 16, 16, 64)
21 conv2_block2_1_relu (None, 16, 16, 64)
22 conv2_block2_2_conv (None, 16, 16, 64)
23 conv2_block2_2_bn (None, 16, 16, 64)
24 conv2_block2_2_relu (None, 16, 16, 64)
25 conv2_block2_3_conv (None, 16, 16, 

b. Make sure all (or at least some) layers in the pretrained ResNet-50 are not trainable.

In [14]:
pretrained.trainable = False
print("Trainable layers:", sum(layer.trainable for layer in pretrained.layers))

Trainable layers: 0


c. Add an output layer adequate to the task we have (predicting a 100 classes).

In [15]:
resnet_cifar = Sequential([
    Input(shape=(32, 32, 3)),
    layers.UpSampling2D(2),
    pretrained,
    layers.GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dense(100, activation='softmax'),
], name='resnet_cifar100')
resnet_cifar.summary()

Model: "resnet_cifar100"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ up_sampling2d (UpSampling2D)    │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2, 2, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,137,956 (92.08 MB)

 Trainable params: 550,244 (2.10 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

d. Compile and train your model.

In [16]:
resnet_cifar.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [17]:
history_resnet = resnet_cifar.fit(
    preprocess_input(train_X.astype('float32')), train_y,
    epochs=3, batch_size=128,
    validation_data=(preprocess_input(test_X.astype('float32')), test_y))

Epoch 1/3

  1/391 ━━━━━━━━━━━━━━━━━━━━ 23:08 4s/step - accuracy: 0.0156 - loss: 8.1535

  2/391 ━━━━━━━━━━━━━━━━━━━━ 3:27 532ms/step - accuracy: 0.0078 - loss: 7.3254

  3/391 ━━━━━━━━━━━━━━━━━━━━ 3:31 546ms/step - accuracy: 0.0208 - loss: 6.6026

  4/391 ━━━━━━━━━━━━━━━━━━━━ 3:44 580ms/step - accuracy: 0.0215 - loss: 6.2954

  5/391 ━━━━━━━━━━━━━━━━━━━━ 3:45 585ms/step - accuracy: 0.0266 - loss: 6.0312

...

388/391 ━━━━━━━━━━━━━━━━━━━━ 1s 550ms/step - accuracy: 0.6744 - loss: 1.1415

389/391 ━━━━━━━━━━━━━━━━━━━━ 1s 551ms/step - accuracy: 0.6744 - loss: 1.1416

390/391 ━━━━━━━━━━━━━━━━━━━━ 0s 551ms/step - accuracy: 0.6745 - loss: 1.1413

391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 550ms/step - accuracy: 0.6745 - loss: 1.1408

391/391 ━━━━━━━━━━━━━━━━━━━━ 258s 661ms/step - accuracy: 0.6745 - loss: 1.1408 - val_accuracy: 0.5246 - val_loss: 1.8893


As you train the model longer, the model will improve its accuracy.

The dataset is balanced, so "accuracy" is an adequate performance metric here.

### Exercise 4

ResNet50 shows that it can help with the CIFAR100 problem.

Let's now check whether it will allow us improve our predictions on the Breast Cancer dataset we worked with in the fist lab.

a. First, reload your images from the same dataset as before. 

Load the images in a data structure, and their correponding labels in another. Make sure that the images are aligned with their labels.

Notice that the dataset has now the images used in the first lab and their corresponding masks. We will need the masks for the segmentation exercises. Do not use them for this classification problem.

In [18]:
data_dir = '../databank/assignment_DS'
classes = ['normal', 'benign', 'malignant']
paths, labels = [], []
for label, c in enumerate(classes):
    files = sorted(f for f in os.listdir(f'{data_dir}/{c}') if '_mask' not in f)
    paths += [f'{data_dir}/{c}/{f}' for f in files]
    labels += [label] * len(files)
images_pil = [Image.open(p).convert('RGB') for p in tqdm(paths)]
print(len(images_pil), np.bincount(labels))

FileNotFoundError: [Errno 2] No such file or directory: '../databank/assignment_DS/normal'

b. Resize our images. Choose The size you wish.

In [ ]:
resized = [im.resize((128, 128)) for im in images_pil]

c. Convert the images to numpy arrays.

In [ ]:
np_images = np.array([np.array(im) for im in resized])

d. Rescale the images.

In [ ]:
np_images = np_images.astype('float32') / 255
print(np_images.shape, np_images.min(), np_images.max())

e. We'll work on the multiclass classification problem. Create a **y** vector with the labels 0 for **normal**, 1 for **benign** and 2 for **malignant**.

In [ ]:
y = np.array(labels)
print(y.shape, np.bincount(y))

f. Let's now create a train and test set from our data.

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(np_images, y, test_size=0.2, shuffle=True, random_state=4)

g. Create a model based on the ResNet50 model and see whether the results of the classification improve compared to the model you developed in the previous lab.

Note that ResNet50 might not be the best pretrained model for all tasks, keras offers other pretrained models. You can try any one of them.

In [ ]:
###Define and compile the model
keras.utils.set_random_seed(42)
base = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base.trainable = False
model_bc = Sequential([
    Input(shape=(128, 128, 3)),
    base,
    layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax'),
], name='resnet_breast_cancer')
model_bc.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_bc.summary()

In [ ]:
###Train the model
train_X_pp = preprocess_input(train_X * 255)
test_X_pp = preprocess_input(test_X * 255)
history = model_bc.fit(train_X_pp, train_y, epochs=20, batch_size=32, validation_data=(test_X_pp, test_y))

In [ ]:
###Plot the training acc and val_acc
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc="lower right")
plt.show()

In [ ]:
###Check its accuracy on the test set
loss, acc = model_bc.evaluate(test_X_pp, test_y, verbose=0)
print(f"ResNet50 test accuracy: {acc:.3f}")
print(confusion_matrix(test_y, model_bc.predict(test_X_pp, verbose=0).argmax(1)))

# From-scratch CNN baseline on the same split, for comparison
keras.utils.set_random_seed(42)
scratch = Sequential([
    Input(shape=(128, 128, 3)),
    Conv2D(16, 3, activation='relu'), MaxPooling2D(),
    Conv2D(32, 3, activation='relu'), MaxPooling2D(),
    Conv2D(64, 3, activation='relu'), MaxPooling2D(),
    Flatten(), Dense(64, activation='relu'), Dense(3, activation='softmax'),
])
scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
scratch.fit(train_X, train_y, epochs=20, batch_size=32, verbose=0)
print(f"From-scratch CNN test accuracy: {scratch.evaluate(test_X, test_y, verbose=0)[1]:.3f}")

## Image Segmentation
Let's now shift to work on image segmentation. 
We have discussed the topic in the lecture, and said that this is a supervised learning problem in which the model learns a mapping between the image and its mask.

### Exercise 5

Let's load the images from our dataset again, this time with their corresponding masks as well.

Make an X data structure with the images, and a Y data structure with the masks.

Check that the images are aligned with their masks, otherwise the training won't work.

In [ ]:
X_pil, Y_pil = [], []
for c in classes:
    for f in sorted(os.listdir(f'{data_dir}/{c}')):
        if '_mask' in f:
            continue
        X_pil.append(Image.open(f'{data_dir}/{c}/{f}'))
        mask_path = next(p for p in (f'{data_dir}/{c}/{f[:-4]}_mask.png', f'{data_dir}/{c}/{f[:-4]}_mask_1.png') if os.path.exists(p))
        Y_pil.append(Image.open(mask_path))
assert all(x.size == m.size for x, m in zip(X_pil, Y_pil))
print(len(X_pil), len(Y_pil))

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j, i in enumerate([0, 200, 600]):
    axes[0, j].imshow(X_pil[i]); axes[0, j].axis('off')
    axes[1, j].imshow(Y_pil[i], cmap='gray'); axes[1, j].axis('off')
plt.show()

### Exercise 6

Convert both images and masks from RGB mode to grayscale, and resize both to a size you see fit.

In [ ]:
SIZE = 128
X_pil = [im.convert('L').resize((SIZE, SIZE), Image.BILINEAR) for im in X_pil]
Y_pil = [m.convert('L').resize((SIZE, SIZE), Image.NEAREST) for m in Y_pil]

### Exercise 7

Convert your images to numpy arrays.
The shape of the masks should be (ne=766, height, width), same goes for the images.

In [ ]:
X_img = np.array([np.array(im) for im in X_pil]).astype('float32') / 255
Y_mask = (np.array([np.array(m) for m in Y_pil]) > 127).astype('float32')
print(X_img.shape, Y_mask.shape, np.unique(Y_mask))

### Exercise 8
We have 766 examples with their corresponding masks, our semantic segmentation model will need more to be able to yield better results.

Apply some data augmentation techniques to generate new data.

Keep in mind that the masks need to be subjected to the same transformations you apply to the images. (if you rotate an image by 25 degrees, the mask should also be rotated with the same value). 

Creating 5 images from each image should give you around 3830 data points with their masks, which should be enough.

Check Scikit-Image for data augmentation functions that you can use, [here](https://scikit-image.org/docs/dev/api/skimage.transform.html)

In [ ]:
tf_shear = AffineTransform(shear=0.2)
transforms = [
    lambda a, order: rotate(a, 25, order=order),
    lambda a, order: np.fliplr(a),
    lambda a, order: np.flipud(a),
    lambda a, order: warp(a, tf_shear, order=order),
]

def augment(X, Y):
    # masks use order=0 (nearest neighbour) so they stay binary
    X_aug = np.concatenate([X] + [np.stack([t(a, 1) for a in X]) for t in transforms])
    Y_aug = np.concatenate([Y] + [np.stack([t(m, 0) for m in Y]) for t in transforms])
    return X_aug.astype('float32'), Y_aug.astype('float32')


Make sure you create a test set as well.

In [ ]:
# Split before augmenting so that no augmented copy of a training image lands in the test set
train_X, test_X, train_y, test_y = train_test_split(X_img, Y_mask, test_size=0.25, shuffle=True, random_state=42)
train_X, train_y = augment(train_X, train_y)
print(train_X.shape, train_y.shape, test_X.shape, test_y.shape)


### Exercise 9

Defining your U-NET architecture below. 

Make sure to use Keras' functional API.

In [ ]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(filters, 3, padding='same', activation='relu')(x)
    return BatchNormalization()(x)

def encoder_block(x, filters):
    skip = conv_block(x, filters)
    return MaxPooling2D()(skip), skip

In [ ]:
def decoder_block(x, skip, filters):
    x = Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = Concatenate()([x, skip])
    return conv_block(x, filters)

In [ ]:
def build_unet(input_shape=(128, 128, 1), base=16):
    inputs = Input(shape=input_shape)
    p1, s1 = encoder_block(inputs, base)
    p2, s2 = encoder_block(p1, base * 2)
    p3, s3 = encoder_block(p2, base * 4)
    p4, s4 = encoder_block(p3, base * 8)
    bottleneck = conv_block(p4, base * 16)
    d1 = decoder_block(bottleneck, s4, base * 8)
    d2 = decoder_block(d1, s3, base * 4)
    d3 = decoder_block(d2, s2, base * 2)
    d4 = decoder_block(d3, s1, base)
    outputs = Conv2D(1, 1, activation='sigmoid')(d4)
    return Model(inputs, outputs, name='unet')

In [ ]:
unet = build_unet()
unet.summary()

### Exercise 10

Compile and train your model. 

This might take quite some time if you're not training on a GPU.

Use the university's Jupyter environment [here](https://jupyter.utwente.nl/) or refer to Google Colab. 

In [ ]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true, y_pred = ops.reshape(y_true, (-1,)), ops.reshape(y_pred, (-1,))
    return (2 * ops.sum(y_true * y_pred) + smooth) / (ops.sum(y_true) + ops.sum(y_pred) + smooth)

def bce_dice_loss(y_true, y_pred):
    return keras.losses.binary_crossentropy(y_true, y_pred) + 1 - dice_coef(y_true, y_pred)

X_tr, X_te = train_X[..., None], test_X[..., None]
Y_tr, Y_te = train_y[..., None], test_y[..., None]

keras.utils.set_random_seed(42)
unet.compile(optimizer='adam', loss=bce_dice_loss, metrics=[dice_coef, keras.metrics.BinaryIoU(name='iou')])
history_unet = unet.fit(X_tr, Y_tr, epochs=10, batch_size=32, validation_data=(X_te, Y_te))

### Exercise 10
Make the predictions based on your trained model, and show for a few examples (3 should be enough), the difference between what your model predicted and the ground truth mask.

In [ ]:
pred = unet.predict(X_te, verbose=0) > 0.5
idx = np.where(Y_te.reshape(len(Y_te), -1).sum(1) > 0)[0][:3]

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, i in enumerate(idx):
    for col, (img, title) in enumerate([(X_te[i], 'Image'), (Y_te[i], 'Ground truth'), (pred[i], 'Prediction')]):
        axes[row, col].imshow(img[..., 0], cmap='gray')
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
plt.show()

## Denoising Autoencoders
### Exercise 11
Let's now practice autoencoder architectures that we discussed in the lecture.

We'll focus on denoising autoencoders.

First, let's load our dataset, and apply some noise to the images. 

You can apply any type of noise you want, refer to [here](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.random_noise)

In [ ]:
X = X_img[..., None]
noised_X = random_noise(X, mode='gaussian', var=0.01).astype('float32')

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j in range(3):
    axes[0, j].imshow(X[j, ..., 0], cmap='gray'); axes[0, j].set_title('Original'); axes[0, j].axis('off')
    axes[1, j].imshow(noised_X[j, ..., 0], cmap='gray'); axes[1, j].set_title('Noised'); axes[1, j].axis('off')
plt.show()

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(noised_X, X, test_size=0.25, shuffle=True, random_state=42)

### Exercise 13
Create your encoder architecture.

This should output the latent space.

In [ ]:
enc_in = Input(shape=(128, 128, 1))
x = Conv2D(32, 3, padding='same', activation='relu')(enc_in)
x = MaxPooling2D()(x)
x = Conv2D(64, 3, padding='same', activation='relu')(x)
x = MaxPooling2D()(x)
x = Conv2D(128, 3, padding='same', activation='relu')(x)
x = MaxPooling2D()(x)
latent = Conv2D(16, 3, padding='same', activation='relu')(x)
encoder = Model(enc_in, latent, name='encoder')
encoder.summary()

### Exercise 14
Create your decoder architecture.

This should input the latent space.

In [ ]:
dec_in = Input(shape=encoder.output.shape[1:])
x = Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu')(dec_in)
x = Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
x = Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
dec_out = Conv2D(1, 3, padding='same', activation='sigmoid')(x)
decoder = Model(dec_in, dec_out, name='decoder')
decoder.summary()

### Exercise 15

Compile the autoencoder now that you have both components.

Train the autoencoder for a number of epochs of your choosing.

In [ ]:
keras.utils.set_random_seed(42)
ae_in = Input(shape=(128, 128, 1))
autoencoder = Model(ae_in, decoder(encoder(ae_in)), name='denoising_autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')
history_ae = autoencoder.fit(train_X, train_y, epochs=30, batch_size=16, validation_data=(test_X, test_y))

### Exercise 16

Use the model to denoise the images on your test set, and show a comparison of your model's work and the ground truth (3 samples will suffice).

In [ ]:
denoised = autoencoder.predict(test_X[:3], verbose=0)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row in range(3):
    for col, (img, title) in enumerate([(test_X[row], 'Noised'), (denoised[row], 'Denoised'), (test_y[row], 'Ground truth')]):
        axes[row, col].imshow(img[..., 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
plt.show()

psnr_noisy = tf.reduce_mean(tf.image.psnr(test_X, test_y, max_val=1.0)).numpy()
psnr_denoised = tf.reduce_mean(tf.image.psnr(autoencoder.predict(test_X, verbose=0), test_y, max_val=1.0)).numpy()
print(f"Mean PSNR - noisy: {psnr_noisy:.2f} dB, denoised: {psnr_denoised:.2f} dB")

## Regular Part

In this part, we wish to explore how to deploy a model and make it accessible to users.

### Exercise 17

Save one of your classification models above in the  **h5** format. 

Create a web app (as simple as possible) where a user can upload an image, the app should then respond with its class.

In [ ]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../app', exist_ok=True)
model_bc.save('../models/breast_cancer_resnet50.h5')

In [ ]:
%%writefile ../app/classifier_app.py
import os
import numpy as np
from flask import Flask, request
from keras.models import load_model
from keras.applications.resnet50 import preprocess_input
from PIL import Image

CLASSES = ['normal', 'benign', 'malignant']
MODEL_PATH = os.path.join(os.path.dirname(__file__), '..', 'models', 'breast_cancer_resnet50.h5')
model = load_model(MODEL_PATH)
app = Flask(__name__)

PAGE = """<h1>Breast ultrasound classifier</h1>
<form method=post enctype=multipart/form-data>
<input type=file name=file accept=image/*>
<input type=submit value=Classify>
</form>"""

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'GET':
        return PAGE
    image = Image.open(request.files['file']).convert('RGB').resize((128, 128))
    probs = model.predict(preprocess_input(np.array(image, dtype='float32')[None]), verbose=0)[0]
    return PAGE + f'<h2>Prediction: {CLASSES[probs.argmax()]} ({probs.max():.1%})</h2>'

if __name__ == '__main__':
    app.run()


In [ ]:
import sys
sys.path.append('../app')
from classifier_app import app as clf_app

with open(paths[0], 'rb') as f:
    response = clf_app.test_client().post('/', data={'file': (f, 'sample.png')}, content_type='multipart/form-data')
print(paths[0], '->', response.get_data(as_text=True).split('<h2>')[-1])

## Advanced Part

In this part, we wish to explore how to deploy a more complex model and make it accessible to users.

### Exercise 18

Save your denoising autoencoder model above in the  **h5** format. 

Create a web app (as simple as possible) where a user can upload an image, the app should then respond with its denoised version.

In [ ]:
autoencoder.save('../models/denoising_autoencoder.h5')

In [ ]:
%%writefile ../app/denoiser_app.py
import base64
import io
import os
import numpy as np
from flask import Flask, request
from keras.models import load_model
from PIL import Image

MODEL_PATH = os.path.join(os.path.dirname(__file__), '..', 'models', 'denoising_autoencoder.h5')
model = load_model(MODEL_PATH, compile=False)
app = Flask(__name__)

PAGE = """<h1>Ultrasound denoiser</h1>
<form method=post enctype=multipart/form-data>
<input type=file name=file accept=image/*>
<input type=submit value=Denoise>
</form>"""

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'GET':
        return PAGE
    image = Image.open(request.files['file']).convert('L').resize((128, 128))
    x = np.array(image, dtype='float32')[None, ..., None] / 255
    out = (model.predict(x, verbose=0)[0, ..., 0] * 255).astype('uint8')
    buffer = io.BytesIO()
    Image.fromarray(out).save(buffer, format='PNG')
    encoded = base64.b64encode(buffer.getvalue()).decode()
    return PAGE + f'<h2>Denoised image</h2><img src="data:image/png;base64,{encoded}" width=384>'

if __name__ == '__main__':
    app.run()


In [ ]:
from denoiser_app import app as ae_app

with open(paths[0], 'rb') as f:
    response = ae_app.test_client().post('/', data={'file': (f, 'sample.png')}, content_type='multipart/form-data')
print(response.status_code, 'data:image/png;base64' in response.get_data(as_text=True))